In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
print("Dataset downloaded to:", path)

In [ ]:
import os

dir_path = os.path.join(path, "PlantVillage")

print("Top-level contents:")
for item in os.listdir(dir_path):
    item_path = os.path.join(dir_path, item)
    if os.path.isdir(item_path):
        print(f"📁 {item}/")
    else:
        print(f"📄 {item}")

In [ ]:
# inside the 'train' folder
train_path = os.path.join(dir_path, "test")

print(f"\nClasses in train folder:")
classes = sorted(os.listdir(train_path))
for i, class_name in enumerate(classes[:10]):
    class_path = os.path.join(train_path, class_name)
    num_images = len(os.listdir(class_path))
    print(f"  {i}: {class_name}")

print(f"\nTotal classes: {len(classes)}")

In [ ]:
from torch.utils.data import Dataset, dataloader

class PotatoDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        # Step 1: Get all class names (folder names)
        self.classes = sorted(os.listdir(root_dir))

        # Step 2: Create a mapping from class name to integer label
        self.class_to_idx = {class_name: i for i, class_name in enumerate(self.classes)}

        # Step 3: Collect all image paths and their labels
        self.image_paths = []
        self.labels = []

        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            class_idx = self.class_to_idx[class_name]

            # Get all image files in this class folder
            for img_name in os.listdir(class_dir):
                img_path = os.path.join(class_dir, img_name)
                self.image_paths.append(img_path)
                self.labels.append(class_idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        # Load image
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')

        # Get label
        label = self.labels[idx]

        # Apply transforms if provided
        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
from PIL import Image

# Define Transform
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])


train_dataset = PotatoDataset(
    root_dir=os.path.join(dir_path, "train"),
    transform=transform
)


test_dataset = PotatoDataset(
    root_dir=os.path.join(dir_path, "test"),
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


# Display 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [20,1200,1621,819,15]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]  # Load image & label

    # Convert tensor to numpy for visualization
    img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) → (H, W, C)

    # Denormalize the image
    img_np = np.clip(img_np, 0, 1)

    # Show image
    axes[i].imshow(img_np)
    axes[i].set_title(f'Class: {label}')
    axes[i].axis('off')

plt.show()

In [ ]:
import torch.nn as nn

# Write your code here
class PotatoCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # first Conv2d
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # [B,32,32,32]
            nn.BatchNorm2d(32),
            nn.ReLU(),

            #second Conv2d layer
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # [B,64,30,30]
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),                  # [B,64,15,15]

            #third Conv2d layer
            nn.Conv2d(64, 128, kernel_size=3, padding=1),   #[B,128,13,13]
            nn.BatchNorm2d(128),
            nn.ReLU(),

            #fourth Conv2d layer
            nn.Conv2d(128, 256, kernel_size=3, padding=1),   #[B,256,11,11]
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2), #[B,256,4,4]

            #fifth Conv2d layer
            nn.Conv2d(256, 512, kernel_size=3, padding=1), #[B,512,2,2]
            nn.BatchNorm2d(512),
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 2 * 2, 256),
            nn.ReLU(),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [ ]:
model = PotatoCNN()
model

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# Write your code here

from tqdm import tqdm

def accuracy_from_logits(logits, labels):
    preds = torch.argmax(logits, dim=1)
    return (preds == labels).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
    # train
    model.train()
    total_loss, total_acc = 0.0, 0.0

    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)

        #Zero the gradients
        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        # Backward pass
        loss.backward()

        # Update parameters
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), labels)

    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
def evaluate(model, loader, criterion):
    # evaluation
    model.eval()
    total_loss, total_acc = 0.0, 0.0

    with torch.no_grad():
        for images in tqdm(loader):
            images, labels = images.to(device), labels.to(device)

            # Get model predictions
            logits = model(images)

            # Calculate loss
            loss = criterion(logits, labels)

            total_loss += loss.item()
            total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
# Write your code here
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr= 0.001)

num_epochs = 5

# Initialize history tracking
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_loss, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), test_loss, label="Test Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_acc, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), test_acc, label="Test Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()

In [ ]:
# Write your code here
